---------------------------------------------------

In [ ]:
# Imports
import torch
import pickle
import glob
import pandas as pd
import numpy as np
import os
import sys
import subprocess
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

In [ ]:
# Create directories
!mkdir -p results work_dir models data

In [ ]:
# Check devices for training
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Copies code from github (https://github.com/moeschl21/TransformerLSR_moeschl.git) into the working directory 
!git clone https://github.com/moeschl21/TransformerLSR_moeschl.git
!cp -r TransformerLSR_moeschl/* /kaggle/working/
!rm -rf TransformerLSR_moeschl

In [ ]:
# Setting up variables
num_sim = 100 # Number of Simulations
num_traj = 100 # Number of patients
timeout = 1000 # Maximum visits
batch_size = 16
num_enc_layer = 2 #2
num_dec_layer = 3 #3
lr = 0.0001 #0.0001
model_size = 64 # Embedding size 64
epoch = 100

In [ ]:
# Copies data from dataset
!cp -r /kaggle/input/datasets/jonasmschl/data-lsr/data_lsr/. /kaggle/working/data/

In [ ]:
########################################################################
############################### TRAINING ###############################
########################################################################

In [ ]:
import logging

logger = logging.getLogger()

for handler in logger.handlers[:]:
    logger.removeHandler(handler)

In [ ]:
# Training with missing Y_1 25%, Y_2 15% and Y3 3%

# Simple Training
for seed in range(1,num_sim + 1):
    get_ipython().run_line_magic(
        "run",
        f"main.py "
        f"--seed {seed} "
        f"--epoch {epoch} "
        f"--batch_size {batch_size} "
        f"--num_enc_layer {num_enc_layer} "
        f"--num_dec_layer {num_dec_layer} "
        f"--lr {lr} "
        f"--model_size {model_size} "
        f"--local "
        f"--data DIVAT_sim_missing_{num_traj}_visit_{timeout}_long_3"
    )

    if seed == num_sim:
        print("done")
    
    logger = logging.getLogger()

    for handler in logger.handlers[:]:
        logger.removeHandler(handler)

In [ ]:
#No parallel training done here